# **E-commerce User Churn Prediction - Exploratory Data Analysis(EDA)**

## **1. Background**
This dataset contains user profile, engagement, transaction and service-related information from an e-commerce platform, along with a Churn indicator identifying whether a user has churned.<br>

**Dataset overview**
- Rows: ~5.6k users
- Target feature: Churn (0 = Retained, 1 = Churned)
- Key feature groups:
  - User demographics
  - Purchase behaviour
  - Engagement activity
  - Service experience
  - Cashback and retention-related signals

## **2. Business Objective**
To identify users with high churn risk so that business can proactively deploy targeted retention strategies to improve user retention, optimise customer lifetime value and reduce unnecessasry retention cost.

## **3. Project Deliverables**
- Explore behavioural patterns associated with user churn
- Identify potential churn drivers through exploratory analysis
- Build a predictive machine learning model to identify high churn risk users
- Generate business insights and recommendations to support retention strategies


## **4. Data Import & Setup**
Import required libraries and load the dataset for exploratory analysis and model development

In [ ]:
# Importing libraries

# Data manipulation
import pandas as pd
import numpy as np
import math as math

# Visualisation
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

# Machine Learning models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Model evaluations
from sklearn.metrics import (accuracy_score, precision_score, recall_score, roc_auc_score, confusion_matrix, 
                            classification_report, roc_curve, f1_score, precision_recall_curve)

# Data preprocessisng
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Utilities
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Importing dataframe
df = pd.read_excel('Dataset/E Commerce Dataset.xlsx',sheet_name='E Comm')

## **5. Data Inspection**
Understand the dataset structure, review summary statistics, and assess data quality through duplicate check

In [ ]:
# Dataframe Overview

print('Dataframe Preview\n')
display(df.head())
print('-' * 100)

print('Dataframe Overview\n')
print(df.info())
print('-' * 100)

print('Dataframe Statistics\n')
display(df.describe().transpose())
print('-' * 100)

# Checking for duplicates based on CustomerID
print('Customer ID duplicates: ',df['CustomerID'].duplicated().sum())

## **6. Data Label Standardisation**
Review and standardise inconsistent categorical labels across the dataset

In [ ]:
# Splitting dataframe columns by data type 
cat_cols = df.select_dtypes(['object','category']).columns
num_cols = df.select_dtypes(['int64','float64']).columns

# Inspecting categorical data labels
print('Categorical fields inspection \n')
for cat in cat_cols:
    print(df[cat].value_counts().sort_index())
    print('-' * 100)

In [ ]:
# Cleaning up categorical inconsistencies 
df['PreferredLoginDevice'] = df['PreferredLoginDevice'].replace('Phone','Mobile Phone')

df['PreferredPaymentMode'] = df['PreferredPaymentMode'].replace({
    'CC':'Credit Card',
    'COD':'Cash',
    'Cash on Delivery':'Cash',
})

df['PreferedOrderCat'] = df['PreferedOrderCat'].replace('Mobile','Mobile Phone')

 **Observation**

Several categorical features contained inconsistent labels
(e.g. "CC" and "Credit Card") hence labels were standardised to reduce category noise before modeling.

## **7. Target Variable Analysis**
Analyse the distribution of the Churn variable to check for class imbalance

In [ ]:
# Visualise Churn distribution

churn_counts = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100

churn_counts.plot(kind='bar', figsize=(15, 8))
plt.ylabel('Count')
plt.title('Churn Distribution')


# Add percentage labels
for i, value in enumerate(churn_counts):
    plt.text(
        i,
        value,
        f'{value}\n({churn_pct.iloc[i]:.0f}%)',
        ha='center'
    )

plt.show()

**Observation**<br>
The dataset is imbalance where 17% of the users are labeled as Churned.

## **8. Univariate Analysis**
Examine the distribution of individual features to underderstand their behaviour, frequency pattern and data spread
### **8.1 Categorical Feature Analysis**

In [ ]:
# Categorical data distribution

rows = math.ceil(len(cat_cols)/2)
fig,axes = plt.subplots(rows,2,figsize=(15,10))
axes = axes.flatten()

for i,col in enumerate(cat_cols):
    ax = sns.countplot(df,y=col,ax=axes[i])

    for p in ax.patches:
        x = p.get_width()
        ax.text(
            x,
            p.get_y() + p.get_height()/2,
            f'{x/len(df):.0%}'
            )

for j in range(len(cat_cols),len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

**Observations**
- 60% of the users are Male
- Mobile phones are the most preferred login device, accounting for 71% of users
- 73% of users prefer to pay via Card
- Electronic gadgets is the most preferred order category, including mobile phone, laptop & accessory.<br> 

### **8.2 Numerical Feature Analysis**

In [ ]:
# Numerical data distribution

plot_cols = [col for col in num_cols if col not in ['CustomerID', 'Churn']]

rows = math.ceil(len(plot_cols)/2)
fig,axes = plt.subplots(rows,2,figsize=(12,12))
axes = axes.flatten()

for i,col in enumerate(plot_cols):
    sns.histplot(df,x=col,ax=axes[i],kde=True)

for j in range(len(plot_cols),len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

**Observations**
- Majority of the users have a tenure of 1 month and below
- Tier 1 cities have the highest number of users, followed by Tier 3 and Tier 2
- Tenure, number of addresses, order amount hike, coupon usage, order count and cashback amount display right-skewed distributions, suggesting varied customer behaviour which is common in e-commerce settings
- Warehouse to home distance and days since last order exhibit bimodal distribution, potentially reflecting distinct geographical delivery segments and varying customer engagement patterns respectively.

## **9. Bivariate Analysis**
Analyse the relationship between individual features and the target variable (Churn) to identify potential churn drivers
### **9.1 Categorical Feature vs Churn**

In [ ]:
# Categorical data distribution by Churn status 

rows = math.ceil(len(cat_cols)/2)
fig,axes = plt.subplots(rows, 2, figsize=(12, 8))
axes = axes.flatten()

for i,cat in enumerate(cat_cols):
    ax = sns.countplot(
        data=df,
        y=cat,
        hue='Churn',
        ax=axes[i],
        )

    for p in ax.patches:
        x = p.get_width()

        if x == 0:
            continue
            
        total = df.groupby(cat).size()
        ax.text(
            x,
            p.get_y() + p.get_height()/2,
            f'{x/len(df):.0%}',
            va = 'center'
        )

for a in range(len(cat_cols),len(axes)):
    axes[a].axis('off')
    
plt.tight_layout()
plt.show()

In [ ]:
# Categorical data breakdown by Churn status

for col in cat_cols:
    col_df = df.groupby([col,'Churn']).agg(
        user_count = ('CustomerID','count')
    )
    col_df['churn_pct'] = ((col_df['user_count']/col_df.groupby(level=0)['user_count'].sum())*100).round(1)
    display(col_df)
    print('-'*100)

**Observations**
- Users primarily login via mobile phones which is also the channel where most churn occurs
- Higher churn rate observed among Singles and cash payment
- Mobile Phone order category has the highest churn, with 27% of users in this category having churned

### **9.2 Numerical Feature vs Churn**

In [ ]:
# Numerical data distribution by Churn status 

rows = math.ceil(len(plot_cols)/2)
fig,axes = plt.subplots(rows,2,figsize=(12,15))
axes = axes.flatten()

for i,col in enumerate(plot_cols):

    ax = axes[i]
    
    sns.histplot(
        data=df,
        x=col,
        hue='Churn',
        ax=ax,
        multiple = 'stack'
    )

for j in range(len(plot_cols),len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Numerical data breakdown by Churn status

for col in plot_cols:

    col_df = df.groupby([col,'Churn']).agg(
        user_count = ('CustomerID','count')
    )
    col_df['churn_pct'] = ((col_df['user_count']/col_df.groupby(level=0)['user_count'].sum())*100).round(1)
    display(col_df)
    print('-'*100)
    

**Observations**
- Higher churn rate observed among lower tenure and days since last order
- Tier 3 cities have the highest churn rate at 21%
- Users who made complaints show a churn rate of 31%, indicating a strong association between complaints and churn behaviour
- Churn users appear to be slightly more engaged, spending marginally higher hours on the app compared to non-churn users
- Churn rate increases as the number of device registered and satistfaction score increases

### **9.3 Correlation Analysis**

In [ ]:
print('Correlation between numerical features\n')

heatmap_cols = num_cols.drop('CustomerID')

plt.figure(figsize=(12,6))
sns.heatmap(df[heatmap_cols].corr(),annot=True,fmt=".2f")
plt.show()

**Observations**
- Complain and Tenure are strong driver of churn
- Positive correlation of 0.25 identified for Complain indicating users who lodged complaints are more likely to churn
- Tenure shows a strong negative correlation of -0.35 suggesting users with shorter tenure have higher churn risk
- Strong positive correlation (0.75) between Coupon Used and Order Count suggests potential multicollinearity between the features
     - To further validate multicollinearity, Variance Inflation Factor (VIF) analysis will be conducted during [feature selection stage](#16.-Feature-Selection)

## **10. Multivariate Analysis**
Explore interactions between multiple features and their relationship with the target variable (Churn)
### **10.1 City Tier vs Churn Analysis**
This section examines whether City Tier is associated with churn behaviour, motivated by the higher churn observed in Tier 3 cities.

In [ ]:
# Exploring data distribution on City Tier level

for cat in cat_cols:
    sns.catplot(df,y=cat,hue='Churn',col='CityTier',kind='count')
    plt.show()

for num in plot_cols:
    sns.displot(df,x=num,hue='Churn',col='CityTier',kind='hist',multiple='stack')
    plt.show()

**Observations**
- Highest user and order count from Tier 1 followed by Tier 3 then Tier 2 city based on descending order
- Tier 1 city most preferred category is Mobile Phones while Tier 3 city prefers Laptop & accessory. However, Mobile Phone remains to have the highest churn rate across both Tier 1 and 3

### **10.2 Number of Devices vs Hours spent vs Churn Analysis**
This section examines whether the higher churn observed among users with higher hours on the app is associated with the number of devices registered.

In [ ]:
# Examining relations between number of device registered, hours spend on app and churn status

df1 = df.groupby(['NumberOfDeviceRegistered','Churn']).agg(
    avg_hours = ('HourSpendOnApp','mean'),
    user_count = ('CustomerID','count'),
)

df1['avg_hours'] = df1['avg_hours'].round(2)
df1['churn_pct'] = ((df1['user_count'] / df1.groupby(level=0)['user_count'].sum())*100).round(0)

display(df1)

**Observation**
- Higher churn rate for users with more devices registered but it does not translate to higher hours spent on app

### **10.3 Complaints vs Preferred Order Category vs Churn Analysis**
This section investigates whether complaint-related churn is concentrated within specific order categories.

In [ ]:
# Examining relations between complains, preferred order category and churn status

df.groupby(['Complain','PreferedOrderCat']).agg(
    user_count = ('CustomerID','count'),
    churn_count = ('Churn','sum'),
    churn_pct = ('Churn','mean')
)
    

**Observation**
- Among users who had made a complaint, those who preferred Mobile Phone exhibit the highest churn rate of 53%

## **11. Key insights**

**1. Early tenure users show highest churn**<br>
High churn is observed among users with tenure of 1 month or below, with approximately 50% of this segment churning, indicating weak New User retention and potential product or service gaps.

**2. Complaint behaviour is a strong churn driver**<br>
Users who lodged complaints exhibit approximately 3x higher churn compared to those without complaints, highlighting complaint behaviour as a strong indicator of churn.

**3. Mobile Phone as top preferred order category drove highest churn, amplified by complaints**<br>
Mobile Phone is the 2nd most preferred order category yet it has the highest churn rate of 27%. Among churn users in this segment, 58% had lodged complaints, suggesting potential negative experience related to product or support issues.

**4. Highly engaged does not imply retention**<br>
Higher number of registered devices and more time spent on app among churn users reflect possible usage friction rather than high quality engagement. 

**5. City Tier impacts churn**<br>
Churn rates vary across city tiers, with Tier 3 exhibiting the highest churn rate of 21% while Tier 1 having the lowest churn rate of 15%, indicating potential disparities in user experience or service quality across cities.

**6. Cash payment users show higher churn**<br>
Cash payment has the highest churn rate of 25% indicating possible payment friction resulting in weaker retention than online payment.

**7. Satisfaction score anomaly**<br>
Unexpected positive relationship between Satisfaction Score and churn rate suggests that the scores may not fully reflect the post-experience dissatisfaction. This metric should therefore be interpreted with caution.

## **12. Data Preprocessing**
Prepare the dataset for modelling by handling data quality issues
### **12.1 Handling Missing Values**

In [ ]:
# Dropping CustomerID as it is a unique identifier and not predictive of churn
df_model = df.drop(columns='CustomerID').copy()

# Splitting model dataframe based on updated data type 
cat_model_cols = df_model.select_dtypes('object').columns
num_model_cols = df_model.select_dtypes(['int64','float64']).columns

# Examine missing values distribution
print('Null Values Overview (%) \n')
null_values = df_model.isnull().mean()
null_values_cols = null_values[null_values > 0].index
print((null_values*100).round(1).sort_values(ascending=False))
print('\nTotal null columns by data type:\n',df_model[null_values_cols].dtypes.value_counts())

In [ ]:
# Summary statistics of the dataset and City Tier breakdown to assess potential geographical differences relevant for imputation
print('DataFrame Statistics Summary')
display(df_model.describe().transpose())
print('-'*100)

city_tier = sorted(df_model['CityTier'].unique())

for t in city_tier:
    city_df = df_model[df_model['CityTier'] == t]
    print('City ',t,' Statistics Summary')
    display(city_df.describe().transpose())
    print('-'*100)

**Observation**
- Median Tenure varies across City Tiers while other features show no meaningful geographical differences

**Imputation Strategy**
- Tenure: median imputation grouped by City Tier
- Other features: global median imputation

To avoid data leakage, all imputation steps will be performed after train-test split.

### **12.2 Handling outliers**

In [ ]:
# Function to populate a summary of features with outliers

def outliers_col(data,num_col):

    result = []
    
    for col in num_col:
        perct_75 = data[col].quantile(0.75)
        perct_25 = data[col].quantile(0.25)
        iqr = perct_75 - perct_25
        upper = perct_75 + 1.5 * iqr
        lower = perct_25 - 1.5 * iqr

        outliers = (data[col] > upper) | (data[col] < lower)

        if outliers.any():
            
            outliers_count = outliers.sum()
            total_count = len(data[col])
            outliers_pct = outliers_count / total_count
    
            result.append(
                {'feature':col,
                 'total_count':total_count,
                 'outlier_count':outliers_count,
                 'outlier_pct':round(outliers_pct*100,2)}
            )

    return pd.DataFrame(result).set_index('feature')

outliers = outliers_col(df_model,num_model_cols)

display(outliers)
print(f'Number of numerical fields with outliers: {len(outliers)}')

In [ ]:
# Inspecting outliers on numerical fields

rows = math.ceil(len(num_model_cols)/3)
fig,axes = plt.subplots(rows,3,figsize=(12,8))
axes = axes.flatten()

for i, cols in enumerate(num_model_cols):    
    sns.histplot(df_model[cols],ax = axes[i],kde=True)
    axes[i].set_title(cols)

for a in range(len(num_model_cols),len(axes)):
    axes[a].axis('off')
    
plt.tight_layout()
plt.show()

rows = math.ceil(len(num_model_cols)/3)
fig,axes = plt.subplots(rows,3,figsize=(12,8))
axes = axes.flatten()

for i, cols in enumerate(num_model_cols):    
    sns.boxplot(df_model[cols],ax = axes[i])
    axes[i].set_title(cols)

for a in range(len(num_model_cols),len(axes)):
    axes[a].axis('off')
    
plt.tight_layout()
plt.show()

**Observations**<br>
Detected outliers are assumed to reflect valid variations in user behaviour rather than data quality issues. Most identified features were therefore retained in their original form as the proportion of outliers and value ranges were relatively small and remained interpretable.<br>

However, log transformation will be applied on Cashback Amount due to its right-skewed distribution and relatively larger value range to reduce skewness and mitigate the influence of large magnitude values during modeling.

## **13. Feature Engineering**
Transform selected features to improve model performance and enhance predictive signal
### **13.1 Variable Transformation**

In [ ]:
# Log transformation for 'CashbackAmount'

outliers_cols = ['CashbackAmount']

for col in outliers_cols:
    df_model[col + '_log'] = np.log1p(df_model[col])

df_model.head()

### **13.2 Categorical Feature Grouping**

In [ ]:
# Payment categories were consolidated into broader behavioural groups (card, cash, digital) to reduce category sparsity and improve model generalisation.
df_model['PreferredPaymentMode'] = df_model['PreferredPaymentMode'].replace({
    'Credit Card':'Card',
    'Debit Card':'Card',
    'E wallet':'Digital',
    'UPI':'Digital'
})

df_model['PreferredPaymentMode'].value_counts()

### **13.3 Dummy encoding**
- One-Hot encoding for PreferredLoginDevice, PreferredPaymentMode, PreferedOrderCat, MaritalStatus
- Label encoding for Gender

In [ ]:
# One-Hot encoding
df_model = pd.get_dummies(df_model,columns=['PreferredLoginDevice','PreferredPaymentMode','PreferedOrderCat','MaritalStatus'],drop_first=True)

# Label encoding
gender_map = {'Female':0,'Male':1}
df_model['Gender'] = df_model['Gender'].map(gender_map)

df_model.head()

## **14. Train-Test Split**
Split the prepared dataset into training and test sets for model development and evaluation

In [ ]:
# Train-test-Validation Split

# Separate the dataset into Features and Target variables  
X = df_model.drop(columns='Churn',axis=1).copy()
y = df_model['Churn']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,random_state=42, stratify=y)

print(f'X_train dataset size: {X_train.shape}')
print(f'y_train dataset size: {y_train.shape}')
print(f'X_test dataset size: {X_test.shape}')
print(f'y_test dataset size: {y_test.shape}')

## **15. Feature Imputation**
Address missing values using imputation strategies derived from observations in [Section 12.1 Handling Missing Values](#12.1-Handling-Missing-Values) to improve data quality and strengthen model's predictive signal

- Tenure: median imputation grouped by City Tier
- Other features: global median imputation

In [ ]:
# Ensure independence
X_train = X_train.copy()
X_test = X_test.copy()

# Imputing Tenure based on City's median
city_median = X_train.groupby('CityTier')['Tenure'].median()

X_train['Tenure'] = X_train['Tenure'].fillna(
    X_train['CityTier'].map(city_median))

X_test['Tenure'] = X_test['Tenure'].fillna(
    X_test['CityTier'].map(city_median))

# Removing Tenure from the list of features with null values
null_impute_cols = null_values_cols.drop('Tenure')

# Imputing the remaining features with median
num_imputer = SimpleImputer(strategy='median')
X_train[null_impute_cols] = num_imputer.fit_transform(X_train[null_impute_cols])
X_test[null_impute_cols] = num_imputer.transform(X_test[null_impute_cols])

# Perform validation check to ensure all missing values have been imputed

null_check = {
    'X_train':X_train,
    'X_test':X_test}

for name,data in null_check.items():
    
    null_count = data.isna().sum()

    if null_count.sum() > 0:
        print(f'Missing value exists in {name}')
        print(null_count[null_count > 0])
    
    else:
        print(f'No missing value in {name}')

## **16. Feature Selection**
Using Variance Inflation Factor (VIF) Analysis to validate the presence of multicollinearity as observed under earlier [Section 9.3 Correlation Analysis](#9.3-Correlation-Analysis) and to guide the selection of optimal features for model training

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Generating list of numerical columns which will be used for modelling
vif_col = X_train.select_dtypes(['float64','int64']).columns.drop('CashbackAmount')

def vif_check(df,vif_cols):
    vif_df = df[vif_cols].copy()
    
    vif_result = []
    
    for i,c in enumerate(vif_df.columns):
        vif_result.append({
            'feature':c,
            'vif':round(variance_inflation_factor(vif_df.values,i),2)
        })
    
    return display(pd.DataFrame(vif_result).sort_values(by='vif',ascending=False))

print('VIF Summary V1')
vif_check(X_train,vif_col)

# Generating list of updated numerical columns which will be used for modelling
vif_col_adj = X_train.select_dtypes(['float64','int64']).columns.drop('CashbackAmount_log')

print('VIF Summary V2')
vif_check(X_train,vif_col_adj)

**Observations**<br>

- Order count and Coupon used exhibit weak multicollinearity which suggests that these 2 features are relatively independent and not strongly explainable by other features
- Log transformed Cashback amount has high VIF of 61, indicating high linear dependency with other features. Hence, original Cashback amount will be used for modelling to reduce multicollinearity and improve model stability
- Although hours spend on app, order amount hike and number of device registered exhibit moderately high VIF, they represent distinct aspects of user engagement. Removal of these features will reduce model expressiveness so it is being retained while being monitored via feature importance

In [ ]:
# Dropping CashbackAmount_log in train and test dataset

X_train.drop(columns='CashbackAmount_log',axis=1,inplace=True)
X_test.drop(columns='CashbackAmount_log',axis=1,inplace=True)

## **17. Feature Scaling**
Apply feature secaling to reduce sensitivity to magnitude differences during model training and improve model performance.

Only continuous numerical features were selected for scaling as these variables are sensitive to magnitude differences.

Discrete or binary numerical features (e.g., Gender, Satisfaction score) were excluded as they represent categorical or ordinal information rather than continuous behavioural measures.

In [ ]:
# Scaling numerical feature with StandardScaler to improve model performance 
sc = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

sc_col = ['Tenure', 'WarehouseToHome', 'HourSpendOnApp',
       'NumberOfDeviceRegistered', 'NumberOfAddress',
       'OrderAmountHikeFromlastYear', 'CouponUsed', 'OrderCount',
       'DaySinceLastOrder', 'CashbackAmount']

X_train_scaled[sc_col] = sc.fit_transform(X_train_scaled[sc_col])
X_test_scaled[sc_col] = sc.transform(X_test_scaled[sc_col])

## **18. Model Training**
Develop and train selected machine learning models to predict user churn

In [ ]:
# Checking if class imblance is consistent between Train & Test dataset
print(f'Train: {y_train.value_counts(normalize=True)}')
print('-'*100)
print(f'Test: {y_test.value_counts(normalize=True)}')

**Observation**<br>
Train and Test dataset both have same proportion of churn users. Due to class imbalance, balanced class weight will be selected for model training.

### **18.1 Logistic Regression Model**
Served as a baseline model for performance benchmark due to its simplicity and interpretability, assuming a linear relationship between features and churn probability.

In [ ]:
lr = LogisticRegression(class_weight='balanced',random_state=42)

lr.fit(X_train_scaled,y_train)

y_lr_fit = lr.predict(X_train_scaled)
y_lr_pred = lr.predict(X_test_scaled)

# Generate Confusion Matrix
print('Confusion Matrix')
sns.heatmap(confusion_matrix(y_test,y_lr_pred),annot=True,fmt='d')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.show()

print('Classification Report')
print(classification_report(y_test,y_lr_pred))

#Generate ROC curve and ROC-AUC score
y_lr_proba = lr.predict_proba(X_test_scaled)[:,1]
roc_auc = roc_auc_score(y_test,y_lr_proba)

fpr,tpr,threshold = roc_curve(y_test,y_lr_proba)

plt.plot(fpr,tpr)
plt.xlabel('False Positive rate')
plt.ylabel('True Positive rate')
plt.title('ROC Curve')
plt.show()

print(f'ROC-AUC Score: {roc_auc:.2f}')

**Observations**<br>
- High ROC-AUC score of 0.88 indicates that model is strong at distinguishing Churn users from non-Churn users
- The model performs well in detecting true churn users, with a recall score of 0.81
- Low precision score (0.44) for Churn detection shows that the model is incorrectly labeling a significant number of non-Churn users as Churn
- To reduce potential additional marketing cost due to incorrect targeting, further adjustment of the classification threshold will be explored

### **18.2 Random Forest**
Selected as a non-linear tree-based model that performs well on imbalanced dataset and its ability to capture complex feature interactions without requiring feature scaling.

In [ ]:
rf = RandomForestClassifier(random_state=42,class_weight='balanced')

rf.fit(X_train,y_train)
y_rf_pred = rf.predict(X_test)
y_rf_proba = rf.predict_proba(X_test)[:,1]

# Generate Confusion Matrix
print('Confusion Matrix')
sns.heatmap(confusion_matrix(y_test,y_rf_pred),annot=True,fmt='d')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.show()

print('Classification Report')
print(classification_report(y_test,y_rf_pred))

#Generate ROC curve and ROC-AUC score
roc_auc_rf = roc_auc_score(y_test,y_rf_proba)
fpr,tpr,thres = roc_curve(y_test,y_rf_proba)
plt.plot(fpr,tpr)
plt.title('ROC Curve')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.show()

print(f'ROC-AUC Score: {roc_auc_rf:.2f}')

**Observations**<br>
- Model demonstrates high ability to distinguish Churn users from non-Churn users, achieving a ROC-AUC score of 0.99
- F1-score of 0.86 indicates a good balance between precision and recall, specifically with precision (0.95) where it suggests a low number of false churn predications

### **18.3 XGBoost**
Used as an advanced boosting model to evaluate potential performance improvement beyond Random Forest and to assess the upper bound of model performance

In [ ]:
xgb = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss'
)

xgb.fit(X_train,y_train)
y_xgb_pred = xgb.predict(X_test)
y_xgb_proba = xgb.predict_proba(X_test)[:,1]

# Generate Confusion Matrix
print('Confusion Matrix')
sns.heatmap(confusion_matrix(y_test,y_xgb_pred),annot=True,fmt='d')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.show()

print('Classification Report')
print(classification_report(y_test,y_xgb_pred))

#Generate ROC curve and ROC-AUC score
roc_auc_xgb = roc_auc_score(y_test,y_xgb_proba)

fpr, tpr, thres = roc_curve(y_test,y_xgb_proba)
plt.plot(fpr,tpr)
plt.ylabel('True Positive Rate')
plt.xlabel('False Positive Rate')
plt.title('ROC Curve')
plt.show()

print(f'ROC-AUC Score: {roc_auc_xgb:.2f}')

**Observations**<br>

- ROC-AUC score of 0.99 indicates strong class separability between between Churn and non-Churn users
- The model achieves a well-balanced performance between Precision and Recall with both metrics above 0.8, exhibiting reliable churn detection

## **19. Pre-Tuning Model Evaluation**
Assess model performance based on initial training results prior to tuning
### **19.1 Baseline Performance & Precision-Recall Analysis**

In [ ]:
# Generate evaluation metrics in a summary table
def get_metrics(model,X_test,y_test):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:,1]

    report = classification_report(y_test,y_pred,output_dict=True)

    return {
    'Precision':round(report['1']['precision'],2),
    'Recall':round(report['1']['recall'],2),
    'F1 Score':round(report['1']['f1-score'],2),
    'ROC-AUC':roc_auc_score(y_test,y_proba).round(2)
    }

summary = pd.DataFrame([
    get_metrics(lr,X_test_scaled,y_test),
    get_metrics(rf,X_test,y_test),
    get_metrics(xgb,X_test,y_test)
],index = ['Logistic Regression','Random Forest','XGBoost'])

print('Baseline Performance Summary')
display(summary)

# Plot Precision-Recall Curve for comparison
precision_lr, recall_lr, thres_lr = precision_recall_curve(y_test,y_lr_proba)
precision_rf,recall_rf, thres_rf = precision_recall_curve(y_test,y_rf_proba)
precision_xgb,recall_xgb,thres_xgb = precision_recall_curve(y_test,y_xgb_proba)

plt.plot(recall_lr,precision_lr,label='Logistic Regression')
plt.plot(recall_rf,precision_rf,label='Random Forest')
plt.plot(recall_xgb,precision_xgb,label='XGBoost')
plt.legend()
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.show()

**Observations**<br>

- Logistic Regression was used as a baseline model to provide interpretability and establish a performance benchmark for churn prediction
- Random Forest and XGBoost outperformed Logistic Regression, achieving a ROC-AUC score of 0.99, indicating a strong non-linear predictive capability in capturing churn behaviour patterns
- XGBoost achieved the highest Recall and F1 score, indicating a slightly better balance between precision and recall compared to Random Forest
- However based on the Precision-Recall Curve, Random Forest demonstrates a stronger Precision-Recall trade-off, suggesting opportunity to explore [threshold tuning](#20.-Model-Tuning-&-Selection) to optimise churn detection performance

### **19.2 Cross-Validation Performance**

To validate the strong ROC-AUC observed in the baseline performance and ensure model stability, Stratified K-Fold cross-validation (k=5) was performed.

This helps will help to confirm whether performance is consistent across different data splits and not driven by a single train-test split.

Logistic Regression was evaluated on scaled features, while tree-based models were evaluated on unscaled data.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
models = {
    "Logistic Regression": lr,
    "Random Forest": rf,
    "XGBoost": xgb
}

for name,m in models.items():
    if name == 'Logistic Regression':
        X_input = X_train_scaled

    else:
        X_input = X_train
    
    kf_scores = cross_val_score(m, X_input, y_train, cv=cv, scoring = 'roc_auc')

    print(f"{name} Mean ROC-AUC:", kf_scores.mean().round(2))
    print(f"{name} Standard Deviation ROC-AUC:", kf_scores.std().round(3))

**Observations**
- All models demonstrate a strong performance with high mean ROC-AUC socres, consistent with the baseline performance results
- Low standard deviation across folds (≤ 0.01 for all models) indicates stable and consistent performance across different data splits, suggesting good generalisation ability
- This confirms that the high ROC-AUC observed is robust and not driven by a single train-test split

## **20. Model Tuning & Selection**
Tune models and select the best performing model to optimise churn prediction performance.

Model Tuning will be performed based on the [business objective](#2.-Business-Objective) of prioritising accurate Churn detection while minimising unnecessary retention efforts on non-churning customers. This enables more efficient allocation of retention resources and optimisation of customer lifetime value.

Using Logistic Regression as baseline model, a recall threshold of >= 0.81 will be applied during threshold tuning. This ensures all models are evaluated under the same Churn detection threshold, allowing a fair comparison  of their precision-recall trade-offs and overall business effectiveness.

In [ ]:
recall_thres = 0.81

idx_lr = np.where(recall_lr[:-1] >= recall_thres)[0]
best_idx_lr = idx_lr[np.argmax(precision_lr[idx_lr])]
best_thres_lr = thres_lr[best_idx_lr]
y_lr_pred_v1 = (y_lr_proba >= best_thres_lr).astype(int)

print(f'\nLogistic Regression Classification report\nwhere threshold = {best_thres_lr:.2f}')
print(classification_report(y_test,y_lr_pred_v1))

idx_rf = np.where(recall_rf[:-1] >= recall_thres)[0]
best_idx_rf = idx_rf[np.argmax(precision_rf[idx_rf])]
best_thres_rf = thres_rf[best_idx_rf]
y_rf_pred_v1 = (y_rf_proba >= best_thres_rf).astype(int)

print(f'\nRandom Forest Classification report\nwhere threshold = {best_thres_rf:.2f}')
print(classification_report(y_test,y_rf_pred_v1))

idx_xgb = np.where(recall_xgb[:-1] >= recall_thres)[0]
best_idx_xgb = idx_xgb[np.argmax(precision_xgb[idx_xgb])]
best_thres_xgb = thres_xgb[best_idx_xgb]
y_xgb_pred_v1 = (y_xgb_proba >= best_thres_xgb).astype(int)

print(f'\nXGBoost Classification report\nwhere threshold = {best_thres_xgb:.2f}')
print(classification_report(y_test,y_xgb_pred_v1))

**Observations**
- **Random Forest is selected as the final model for churn detection** as it achieves the best overall performance after applying a recall threshold of >= 0.81
  - Highest precision and recall of 0.94 and 0.82 respectively indicates model is most effective in correctly identifying Churn users while minimising false churn predictions

## **21. Key Churn Drivers**
The top churn drivers are derived from the final selected model, Random Forest, using feature importance scores. These features represent the strongest predictors of churn risk identified by the model and are further backed by insights from exploratory data analysis.

In [ ]:
# Examining features importance
feature_impt = pd.DataFrame({
    'feature' : X_train.columns,
    'importance' : rf.feature_importances_.round(2)
}).sort_values(by='importance',ascending=False)

top_feature_impt = feature_impt.head(10)
bot_feature_impt = feature_impt.tail(10)

fig,axes = plt.subplots(1,2,figsize=(16,6))
axes[0].barh(top_feature_impt['feature'],top_feature_impt['importance'])
axes[0].set_title("Most important features")
axes[0].set_xlabel("Importance")
axes[0].invert_yaxis()

axes[1].barh(bot_feature_impt['feature'],bot_feature_impt['importance'])
axes[1].set_title("Least important features")
axes[1].set_xlabel("Importance")

plt.tight_layout()
plt.show()

print('-'*100)

# Churn rate by CashbackAmount bin
cash_abv150 = df_model[df_model['CashbackAmount'] >= 150]
cash_below150 = df_model[df_model['CashbackAmount'] < 150]

print('Cashback Churn rate')
print('Cashback >=150: ',cash_abv150['Churn'].mean().round(2))
print('Cashback <150: ',cash_below150['Churn'].mean().round(2))
print('-'*100)

# Churn rate by WarehouseToHome distance
df_model.groupby('WarehouseToHome')['Churn'].mean().plot()
plt.title('Churn rate by Warehouse Distance')
plt.show()

**Observations**
- **Behavioural Drivers**
    - Tenure: Approximately 50% of users with tenure of 1 month or below exhibit high churn risk, indicating weaker retention among new users and highlighting early lifecycle engagement gaps
    - Cashback Amount: Users with cashback ≥ 150 show ~50% lower churn rate compared to lower cashback users, suggesting a strong association between incentive engagement and retention
    - Complaint Behaviour: Users with complaints show approximately 3x higher churn risk, indicating product & service experience as a key retention factor
    - Days Since Last Order: 37% of churned users had their last order within 0–1 days prior to churn, suggesting churn can occur shortly after recent engagement rather than only after long inactivity<br><br>
      
- **Operational Drivers**
     - Warehouse-to-Home Distance: Churn risk is higher in the 20-30km range, suggesting logistics factors may be associated with user retention


## **22. Recommendations**
- Strengthen early lifecycle engagement by targeting new users within the first month of sign-up with personalised communications based on purchase preference to improve activation and retention
  
- Improve post-purchase experience by sending follow-up communications such as product usage guidance and support information to help users better navigate issues, potentially reducing complaint-related churn risk

- Optimise incentive strategy by targeting high churn-risk users with tailored cashback campaigns to encourage repeat engagement and improve retention

- Monitor and improve delivery experience in mid-distance zones (approximately 20–30km from warehouse) where higher churn risk is observed to better understand and address potential service friction